In [ ]:
# ANALYSIS ITRANSFORMER
# SEARCH WORDS :
# YEARLY_ITR_ESTIMATION
# ITR_FITTED_TARGET_SPACE_PERFORMANCE
# ITR_SCORE_CALIBRATION
# ITR_THRESHOLD_CALIBRATION
# ITR_THRESHOLD_TRADER_CALIBRATION

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from pathlib import Path
from arch.utility.exceptions import ConvergenceWarning


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

sys.modules.pop("srcB", None)


# =============================================================================================================
# ======================================= LOAD BASE OUTPUTS ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl","rb") as f:
    outputs_by_year = pickle.load(f)

In [ ]:
# =============================================================================================================
# ======================================= IMPORTS FROM SRC ====================================================
# =============================================================================================================

from srcB.market import *
from srcB.helpers import *

In [ ]:
# =============================================================================================================
# ======================================= READ D_BTC AND DF ===================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")

In [ ]:
# =============================================================================================================
# ======================================= SET INDEX TO DATETIME ===============================================
# =============================================================================================================

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(pd.to_datetime(d_btc_raw_by_year[y]["date_time"]))
    df_raw_by_year[y] = df_raw_by_year[y].set_index(pd.to_datetime(df_raw_by_year[y]["date_time"]))

In [ ]:
# =============================================================================================================
# ======================================= LOAD X_BY_YEAR ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2

Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]

In [ ]:
# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]

In [ ]:
# =============================================================================================================
# ======================================= BORROW RFR COLUMN ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/ready_data/d_btc_by_year_all.pkl", "rb") as f:
    d_btc_by_year = pickle.load(f)

for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(d_btc_raw_by_year[y].index, method="ffill")

In [ ]:
# =============================================================================================================
# ======================================= ITR PARAMETERS ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters/itr_best_params.pkl", "rb") as f:
    best_itr_params = pickle.load(f)

VAL_FRACTION = 0.20
EPOCHS = 40
PATIENCE = 6
SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= YEARLY_ITR_ESTIMATION ================================================
# =============================================================================================================

sys.modules.pop("srcB.itr_model", None)
from srcB.itr_model import *

itr_bundle_by_year = {}
itr_targets_by_year = {}
itr_output_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()
    X_df["garch"] = garch_df.reindex(X_df.index)

    itr_output, itr_bundle, itr_targets = itr_mfiv(
        X_df=X_df,
        option_df=df_raw_by_year[y],
        time_df=d_btc_raw_by_year[y],
        sequence_length=168,
        forward_hours=24,
        delta_target=0.20,
        val_fraction=VAL_FRACTION,
        d_model=best_itr_params["d_model"],
        n_heads=best_itr_params["n_heads"],
        e_layers=best_itr_params["e_layers"],
        d_ff=best_itr_params["d_ff"],
        dropout=best_itr_params["dropout"],
        epochs=EPOCHS,
        batch_size=best_itr_params["batch_size"],
        learning_rate=best_itr_params["learning_rate"],
        weight_decay=best_itr_params["weight_decay"],
        patience=PATIENCE,
        seed=SEED
    )
    itr_output_by_year[y] = itr_output
    itr_bundle_by_year[y] = itr_bundle
    itr_targets_by_year[y] = itr_targets

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        itr_q = itr_output.reindex(q_index)
        outputs_by_year[y][q] = outputs_by_year[y][q].drop(columns=itr_q.columns, errors="ignore")
        outputs_by_year[y][q] = pd.concat([outputs_by_year[y][q], itr_q], axis=1)

In [ ]:
# =============================================================================================================
# ================================ ITR_FITTED_TARGET_SPACE_PERFORMANCE =========================================
# =============================================================================================================

target_cols = [
    "target_long_straddle",
    "target_short_straddle",
    "target_long_strangle",
    "target_short_strangle"
]

rows = []

for y in YEARS:
    test = pd.concat([
        itr_output_by_year[y][["itr_action"]],
        itr_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    realized = pd.Series(0.0, index=test.index)

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        mask = test["itr_action"] == action
        realized.loc[mask] = test.loc[mask, f"target_{action}"]

    sharpe = realized.mean() / realized.std()

    rows.append({
        "year": y,
        "n": len(realized),
        "mean_target": realized.mean(),
        "target_sharpe": sharpe,
        "profitable": (realized > 0).mean()
    })

target_sharpe = pd.DataFrame(rows)
print(target_sharpe)

   year     n  mean_target  target_sharpe  profitable
0  2019  2194     0.004192       0.170536    0.380583
1  2020  4093     0.002767       0.092419    0.414610
2  2021  5008     0.006728       0.272919    0.619010


In [ ]:
# =============================================================================================================
# ======================================= ITR_SCORE_CALIBRATION ================================================
# =============================================================================================================

rows = []

for y in YEARS:
    test = pd.concat([
        itr_output_by_year[y][["itr_action", "itr_best_score"]],
        itr_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    test = test[test["itr_action"] != "no_trade"].copy()
    test["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = test["itr_action"] == action
        test.loc[m, "target_pnl"] = test.loc[m, f"target_{action}"]

    test["score_bin"] = pd.qcut(
        test["itr_best_score"],
        5,
        labels=False,
        duplicates="drop"
    )

    x = (
        test.groupby("score_bin")
        .agg(
            n=("target_pnl", "size"),
            mean_score=("itr_best_score", "mean"),
            mean_target=("target_pnl", "mean"),
            sharpe=("target_pnl", lambda z: z.mean() / z.std()),
            profitable=("target_pnl", lambda z: (z > 0).mean())
        )
        .reset_index()
    )

    x["year"] = y
    rows.append(x)

score_check = pd.concat(rows, ignore_index=True)

print(score_check[[
    "year", "score_bin", "n", "mean_score",
    "mean_target", "sharpe", "profitable"
]].to_string(index=False))

 year  score_bin   n  mean_score  mean_target    sharpe  profitable
 2019          0 310    0.000651     0.002259  0.112174    0.467742
 2019          1 309    0.002099    -0.000717 -0.043249    0.504854
 2019          2 310    0.003725     0.001161  0.063307    0.516129
 2019          3 309    0.006156     0.003538  0.142297    0.550162
 2019          4 310    0.012641     0.023438  0.498505    0.658065
 2020          0 645    0.000686    -0.000881 -0.047201    0.502326
 2020          1 644    0.001991    -0.000193 -0.009642    0.495342
 2020          2 644    0.003462     0.000338  0.017470    0.526398
 2020          3 644    0.005444     0.002941  0.111312    0.546584
 2020          4 644    0.011912     0.015383  0.253323    0.563665
 2021          0 932    0.001153     0.000399  0.021853    0.526824
 2021          1 931    0.003055     0.003206  0.154171    0.595059
 2021          2 931    0.005106     0.005929  0.253451    0.678840
 2021          3 931    0.007695     0.010038  0

In [ ]:
# =============================================================================================================
# ======================================= ITR_THRESHOLD_CALIBRATION ============================================
# =============================================================================================================

tests = {}

for y in YEARS:
    x = pd.concat([
        itr_output_by_year[y][["itr_action", "itr_best_score"]],
        itr_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    x = x[x["itr_action"] != "no_trade"].copy()
    x["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = x["itr_action"] == action
        x.loc[m, "target_pnl"] = x.loc[m, f"target_{action}"]

    tests[y] = x

scores_all = pd.concat([tests[y]["itr_best_score"] for y in YEARS])

rows = []

for q in [0, .20, .40, .50, .60, .70, .80, .85, .90]:

    threshold = scores_all.quantile(q)

    for y in YEARS:
        x = tests[y][tests[y]["itr_best_score"] >= threshold]
        sharpe = x["target_pnl"].mean() / x["target_pnl"].std()

        rows.append({
            "quantile": q,
            "threshold": threshold,
            "year": y,
            "n": len(x),
            "mean_target": x["target_pnl"].mean(),
            "sharpe": sharpe
        })

threshold_check = pd.DataFrame(rows)

print(threshold_check.to_string(index=False))

 quantile    threshold  year    n  mean_target   sharpe
     0.00 4.945323e-07  2019 1548     0.005942 0.204249
     0.00 4.945323e-07  2020 3221     0.003516 0.104298
     0.00 4.945323e-07  2021 4656     0.007236 0.283845
     0.20 1.673538e-03  2019 1177     0.007201 0.229256
     0.20 1.673538e-03  2020 2409     0.004989 0.133744
     0.20 1.673538e-03  2021 3954     0.008515 0.323185
     0.40 3.294162e-03  2019  842     0.010455 0.297903
     0.40 3.294162e-03  2020 1672     0.007032 0.165037
     0.40 3.294162e-03  2021 3141     0.010180 0.367061
     0.50 4.207677e-03  2019  689     0.012351 0.328068
     0.50 4.207677e-03  2020 1320     0.008961 0.191415
     0.50 4.207677e-03  2021 2704     0.011029 0.387616
     0.60 5.296758e-03  2019  551     0.014533 0.358666
     0.60 5.296758e-03  2020  981     0.011328 0.214795
     0.60 5.296758e-03  2021 2238     0.012203 0.418781
     0.70 6.618372e-03  2019  418     0.018452 0.425199
     0.70 6.618372e-03  2020  685     0.014863 0

In [ ]:
# =============================================================================================================
# ======================================= ITR SCORE DISTRIBUTION ===============================================
# =============================================================================================================

for y in YEARS:
    x = tests[y]["itr_best_score"]

    print(
        y,
        "n:", len(x),
        "min:", x.min(),
        "q25:", x.quantile(.25),
        "q50:", x.quantile(.50),
        "q75:", x.quantile(.75),
        "q90:", x.quantile(.90),
        "max:", x.max()
    )

2019 n: 1548 min: 2.1599699e-06 q25: 0.0017470086459070444 q50: 0.0036883215652778745 q75: 0.006934102973900735 q90: 0.010957743041217325 max: 0.029457005
2020 n: 3221 min: 4.945323e-07 q25: 0.0016593618784099817 q50: 0.0034569846466183662 q75: 0.0060623218305408955 q90: 0.009352407418191433 max: 0.049378432
2021 n: 4656 min: 1.1792552e-05 q25: 0.002596994338091463 q50: 0.005076402798295021 q75: 0.008446440799161792 q90: 0.012525260914117098 max: 0.0317332


In [ ]:
# =============================================================================================================
# ======================================= SAVE ITR BUNDLES =====================================================
# =============================================================================================================

os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/itr", exist_ok=True)
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/itr/itr_bundle_by_year.pkl", "wb") as f:
    pickle.dump(itr_bundle_by_year, f)

In [ ]:
# =============================================================================================================
# ======================================= PREPARE TRADER INPUT ================================================
# =============================================================================================================

for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["rv"] = Xq_by_year[y][q]["rv"].reindex(
            outputs_by_year[y][q].index
        )
        outputs_by_year[y][q]["MFIV_24"] = Xq_by_year[y][q]["MFIV_24"].reindex(
            outputs_by_year[y][q].index
        )

In [ ]:
# =============================================================================================================
# ======================================= ITR TRADER ADAPTER ===================================================
# =============================================================================================================
# iTransformer signals are not altered;
# it only copies them into column names the existing trader expects.
for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["tcn_action"] = outputs_by_year[y][q]["itr_action"]
        outputs_by_year[y][q]["tcn_best_score"] = outputs_by_year[y][q]["itr_best_score"]

In [ ]:
# =============================================================================================================
# ======================================= CACHE FOR WARMUP ====================================================
# =============================================================================================================

CACHE_YEARS = [2019, 2020, 2021]

trader_cache = {
    y: pd.concat(
        [
            outputs_by_year[yy][q]
            for yy in CACHE_YEARS
            if yy <= y
            for q in QUARTERS
        ]
    ).sort_index()
    for y in CACHE_YEARS
}

In [ ]:
# =============================================================================================================
# ================================ ITR_THRESHOLD_TRADER_CALIBRATION ============================================
# =============================================================================================================

sys.modules.pop("srcB.trader_testing_tcn", None)
from srcB.trader_testing_tcn import testing_execute_trading_strategy

THRESHOLDS = [0.0]

for q in [.25, .50, .75, .90]:
    threshold = min(
        tests[y]["itr_best_score"].quantile(q)
        for y in YEARS
    )
    THRESHOLDS.append(float(threshold))

THRESHOLDS = sorted(set(THRESHOLDS))

print("iTransformer trader thresholds:", THRESHOLDS)

rows = []

for threshold in THRESHOLDS:
    for y in YEARS:

        market = Market(
            option_df=df_raw_by_year[y],
            time_df=d_btc_raw_by_year[y],
            starting_cash=100_000,
            im_multiplier=0.15,
            mm_buffer_rate=0.05,
        )

        snapshots, greek_tests = testing_execute_trading_strategy(
            bt_year=y,
            bt_quarter=None,
            model="tcn_mfiv",
            outputs_by_year=outputs_by_year,
            market=market,
            trader_cache=trader_cache,
            run_by="year",
            tcn_score_threshold=threshold
        )

        equity = pd.Series(
            [s["equity"] for s in snapshots],
            index=[s["time"] for s in snapshots]
        )

        returns = equity.pct_change().dropna()
        sharpe = returns.mean() / returns.std() * np.sqrt(24 * 365)

        rows.append({
            "threshold": threshold,
            "year": y,
            "trades": len(greek_tests),
            "sharpe": sharpe
        })

threshold_trader_check = pd.DataFrame(rows)

print(threshold_trader_check.to_string(index=False))

iTransformer trader thresholds: [0.0, 0.0016593618784099817, 0.0034569846466183662, 0.0060623218305408955, 0.009352407418191433]
 threshold  year  trades   sharpe
  0.000000  2019     220 0.180930
  0.000000  2020     257 0.917572
  0.000000  2021     262 1.359614
  0.001659  2019     200 0.520749
  0.001659  2020     238 0.759389
  0.001659  2021     262 1.474165
  0.003457  2019     150 0.649047
  0.003457  2020     197 0.791780
  0.003457  2021     247 1.746178
  0.006062  2019     108 0.645623
  0.006062  2020     129 0.260524
  0.006062  2021     216 1.562297
  0.009352  2019      70 0.712594
  0.009352  2020      72 0.579466
  0.009352  2021     139 1.574006
